# 09 · Example-level arm, done properly: atypical examples
### Machine Unlearning is Extinction, Not Forgetting

**Why this notebook.** Notebook 07 forgot a *random* 10 % of the training set. Those are typical images: the retrained
model already gets 74 % of them right through generalisation, so only a thin layer of memorisation separates it from
the original, and **no method reached the matching target** (run #9). Still, the two methods that did forget part of
it (RL, SalUn) relapsed strongly under retain fine-tuning (70 % of the forgetting came back, run #10 preview), while
class-level forgetting never relapsed. This notebook makes that comparison clean.

**What changes: the forget set.** As in "From Dormant to Deleted" (Siddiqui et al. 2025), we forget **atypical**
examples: images with the lowest **C-score** (Jiang et al. 2021), the expected accuracy of models that did not train
on the image. Atypical images are learned only by memorisation: the original model gets them right, a retrained model
mostly does not. The gap between the two is large, so "forgotten" and "came back" are unambiguous.

| Stage | What | Jobs | Time on T4 x2 |
|---|---|---|---|
| **H** head-bias check, v2 | as notebook 07, plus the one-constant shift set on half of the test set and measured on the other half, so the false-positive rate is really 1 % (or 5 %) | 360 | ~15 min |
| **A** retrain | from scratch without each forget set: 3 seeds × 3 forget sets + 1 pilot set | 10 | ~45 min |
| **B** calibrate | denser grid per method on the pilot forget set | 45 | ~45 min |
| **C** unlearn | 5 methods × 9 forget sets (+ 5 pilot), up to 4 candidates each until matched | 50 | ~1.5-2 h |
| **D** relapse tests | identical to notebooks 04 and 07 | 63 | ~1.3 h |

### The design in one paragraph
Take the **5 % least consistent** training images (2,500, the "atypical pool"). Each forget set is **500 images drawn
from that pool** for (seed, draw), so the 9 forget sets differ. An unlearned model counts as **matched** if its accuracy
on its 500 forgotten images is within **±5 points** of its retrained model's (500 images give about ±2 points of noise)
and its test accuracy is within 3 points. **Primary score: accuracy on the forgotten images** (as in D2D). Relapse = that
accuracy rising after a manipulation, beyond what the retrained model gains.

### Pre-registered (2026-10-01, before any atypical data)
XH1 primary: forget accuracy after 5 epochs of retain fine-tuning (BN updated) minus before, minus the retrained
model's, one-sided Wilcoxon per method (matched models). B1: relapse fraction, example > class, one-sided Mann-Whitney.

## Kaggle settings for this notebook

Set these in the right-hand **Settings** panel (in the editor: *View → Show sidebar* if it is hidden).

| Setting | Value | Why |
|---|---|---|
| **Accelerator** | **GPU T4 x2** | Two GPUs: `ext_utils` runs one independent worker per GPU, so two models train at the same time. Your weekly quota is counted in session hours, not per GPU, so T4 x2 roughly doubles what you get per hour. (A single P100 also works: the code detects one GPU and uses one worker.) |
| **Internet** | **On** | Needed to download CIFAR-100 (~170 MB) and, in notebook 04, CIFAR-100-C, unless you attach them as datasets. Internet requires a phone-verified Kaggle account. |
| **Persistence** | Files only (optional) | Keeps `/kaggle/working` between *interactive* sessions, so you don't lose files if the browser disconnects. Not needed for background runs. |
| **Environment** | *Always use latest* for the first run, then **Pin to original environment** | Pinning freezes the library versions (PyTorch etc.), so a Kaggle image update can't break a half-finished experiment. |
| **Language** | Python | |

**Inputs to attach** (*Add Input* → *Your Work* / *Your Datasets*):

| Input | Needed by | Why |
|---|---|---|
| `ext-code` (**version 1.4**: `ext_utils.py` **and** `cifar100-cscores-orig-order.npz`) | all stages | Adds the atypical design and the held-out shift measurement; the `.npz` holds the C-scores (downloaded if missing). Class-level code unchanged. |
| CIFAR-100 (`fedesoriano/cifar100`) | all stages | Check that setup prints `CIFAR-100: found at …`. |
| `ext-01-originals` | all stages | Starting point of every unlearning run; reference for the head-bias check. |
| `ext-02-retrain`, `ext-03-unlearning` | stage H only | The class-level models whose output layer is re-checked. |
| *(when resuming)* this notebook's previous output (`ext-09-atypical`) | – | Skips finished jobs. |

**Internet must be On**: the C-scores (≈ 0.4 MB, `cifar100-cscores-orig-order.npz` from the authors' site) are
downloaded once. If the download is blocked, upload that file as a dataset and attach it.

**How to run.** Click **Save Version → Save & Run All (Commit)**. The notebook then runs in the background
for up to 12 hours with the browser closed. When it finishes, its files appear under *Output*, and the next
notebook attaches them as an input. 

**If a run stops early** (time budget or error), nothing is lost. Every finished job is written to disk
immediately. Attach this notebook's own latest output as an input and run it again: finished jobs are
skipped and their files are carried over.

## Setup: import the shared library

This cell finds `ext_utils.py` and imports it as `U`. All the real code (data pipeline, model, unlearning
methods, tests, job runner) lives there, so every notebook uses *identical* code.

`U.env_report()` then prints:
- the Python and PyTorch versions,
- which GPUs are visible (you should see **two Tesla T4** with the T4 x2 accelerator),
- free disk space,
- whether the internet is reachable,
- which inputs are attached.

**Check this output before going further.** If it says `GPU: none found`, the accelerator is off. If it
says `internet: OFF` and no CIFAR-100 dataset is attached, data loading will fail.

In [ ]:
# Locate ext_utils.py (from an attached dataset, a utility script, or the working folder) and import it.
import os, sys, glob
candidates = (glob.glob("/kaggle/working/ext_utils.py")
              + glob.glob("/kaggle/usr/lib/**/ext_utils.py", recursive=True)
              + glob.glob("/kaggle/input/**/ext_utils.py", recursive=True)
              + glob.glob(os.path.join(os.getcwd(), "ext_utils.py"))
              + glob.glob(os.path.join(os.getcwd(), "..", "ext_utils.py")))
if not candidates:
    raise FileNotFoundError(
        "ext_utils.py not found. Upload it as a private Kaggle Dataset (e.g. 'ext-code') and attach it "
        "with 'Add Input' -> 'Your Datasets' (see the Kaggle settings section above).")
sys.path.insert(0, os.path.dirname(os.path.abspath(candidates[0])))
import ext_utils as U
print("ext_utils loaded from:", U.UTILS_FILE)
U.env_report()

## Configuration

Shared settings must equal notebook 01's (checked). The example-level settings for this design:

| Parameter | Value | Why |
|---|---|---|
| `xl_design` | `"atypical"` | Forget sets drawn from the least consistent images (C-score), not at random. |
| `xl_pool_frac` | 0.05 | The atypical pool: the 5 % lowest C-scores (2,500 images). |
| `xl_n` | 500 | Images per forget set (1 % of the training set, as D2D's dataset-wide setting). |
| `xl_draws`, `xl_pilot_draw` | `[0, 1, 2]`, 9 | 3 forget sets per original model = 9 units per method; draw 9 only for calibration. |
| `xl_match_gap` | 0.05 | Matching: forget accuracy within ±5 points of the retrained model's. |
| `xl_sv_shots` | `[10, 100]` | Savings: relearn from 10 or 100 of the forgotten images (D2D's numbers) and score the other forgotten images. |
| `max_tries` | 4 | One more candidate per model than before, from a denser grid. |
| relapse tests | as notebooks 04 and 07 | Identical procedures, so class and example level compare directly. |

In [ ]:
CFG = U.make_cfg(
    smoke=False,
    seeds=[0, 1, 2],
    forget_classes=["maple_tree", "tiger", "rose", "shark", "bus",
                    "apple", "chair", "butterfly", "girl", "castle"],
    pilot_class="lamp",
    norm="bn",
    width=64, epochs=40, batch_size=256, lr=0.1, pct_start=0.25,
    momentum=0.9, nesterov=True, weight_decay=5e-4, label_smoothing=0.0,
    amp=True, channels_last=True, cudnn_benchmark=True, eval_batch_size=1024,
    ckpt_fp16=True, retain_eval_subset=5000,
    methods=["FT", "NegGrad+", "RL", "SCRUB", "SalUn"],
    forget_batch_size=64, clip_grad=5.0, match_retain_gap=0.03, max_tries=4,
    aba_classes=["maple_tree", "tiger", "rose", "shark", "bus"], aba_context="gray",
    mctx_classes=["maple_tree", "tiger", "rose", "shark", "bus"], mctx_contexts=["clean", "gray", "blur"],
    recovery_bn_modes=["update", "frozen"], bn_recal_sizes=[10, 100, 1000, 10000],
    sr_epochs=5, sr_lr=0.01, sr_noise=[0.005, 0.01, 0.02, 0.05], sr_noise_reps=3,
    sr_quant_bits=[8, 6, 4], sr_prune=[0.2, 0.5, 0.7], ctx_synthetic=["gray", "blur", "noise", "contrast"],
    sv_steps=100, sv_lr=0.01, sv_replay_bs=128, sv_eval_steps=[0, 1, 2, 5, 10, 20, 50, 100], collapse_frac=0.5,
    # example-level arm, atypical design
    xl_design="atypical", xl_pool_frac=0.05, xl_n=500,
    xl_draws=[0, 1, 2], xl_pilot_draw=9, xl_match_gap=0.05, xl_sv_shots=[10, 100],
    time_budget_h=11.0, n_workers=None, reuse_ckpt=True, raise_on_error=True, poll_seconds=60,
)
RUN_H, RUN_A, RUN_B, RUN_C, RUN_D = True, True, True, True, True
U.check_experiment_config(CFG)
assert U.VERSION >= "1.4", "Attach ext-code version 1.4."

## Check the forget sets before spending GPU

Loads the C-scores (downloads them once) and builds the pilot forget set on the CPU. Expect 500 images, C-scores near
the bottom of the range, spread over many classes. If this cell fails, fix it before the GPU stages run.

In [ ]:
import numpy as np
_D = U.load_data(CFG); _G = U.gpu_data(_D, CFG, "cpu")
_sc = U.cscores(CFG, _G)
_t = U.xl_target(_G, CFG, CFG["seeds"][0], CFG["xl_pilot_draw"])
_f = _t.fidx.numpy()
print(f"C-scores: {len(_sc)} images, median {np.median(_sc):.3f}; atypical pool cut-off {np.sort(_sc)[int(CFG['xl_pool_frac'] * len(_sc)) - 1]:.3f}")
print(f"pilot forget set: {len(_f)} images, C-score mean {_sc[_f].mean():.3f} (max {_sc[_f].max():.3f}), {len(set(_D['ytr'][_f]))} classes")
del _D, _G

## Stage H: head-bias check, v2

Same as notebook 07, plus `shift_recall_fpr1` / `shift_recall_fpr5`: the constant is set on the even-numbered test images
so that 1 % (or 5 %) of their other-class images flip, and recall is measured on the odd-numbered ones. Replaces the
1.3 results (older rows are ignored automatically).

In [ ]:
if RUN_H:
    jobs = U.jobs_headbias(CFG)
    print(f"{len(jobs)} models for the head-bias check")
    U.launch("headbias", jobs, CFG)

## Stage A: retrained references

One model per (seed, forget set), trained from scratch on the other 49,500 images with the original recipe and seed.
Its accuracy on the 500 forgotten images is the target.

In [ ]:
if RUN_A:
    U.launch("xl_retrain", U.jobs_xl_retrain(CFG), CFG)
    xr = U.load_df("xl_retrain", CFG)
    if len(xr):
        xr = xr[xr["design"] == "atypical"] if "design" in xr else xr.iloc[0:0]
        display(xr[["seed", "draw", "n_forget", "forget_acc", "test_acc", "mem_gap"]].round(4))

## Stage B: calibration on the pilot forget set

In [ ]:
if RUN_B:
    jobs = U.jobs_xl_calibrate(CFG)
    print(f"{len(jobs)} calibration jobs")
    U.launch("xl_calibrate", jobs, CFG)
    xl_candidates, xl_table = U.xl_select_candidates(CFG)
    display(xl_table.round(4))
xl_candidates = U.load_xl_candidates(CFG)
for m, c in xl_candidates.items():
    print(f"{m:9s} tries in order: {c}")

**How to read the calibration table.** For each method at least one row should be `matched = True`. If a method has
none, it still runs with its closest settings and is reported as unable to reach retraining-level forgetting here.

## Stage C: unlearning

In [ ]:
if RUN_C:
    U.launch("xl_unlearn", U.jobs_xl_unlearn(CFG, xl_candidates, pilot=True) + U.jobs_xl_unlearn(CFG, xl_candidates), CFG)

## Stage D: relapse tests

In [ ]:
if RUN_D:
    jobs = U.jobs_xl_extinction(CFG)
    print(f"{len(jobs)} models to test")
    U.launch("xl_extinction", jobs, CFG)

## Quick look

In [ ]:
import numpy as np, pandas as pd
def _atyp(d): return d[d["design"] == "atypical"] if len(d) and "design" in d else d.iloc[0:0]
xu = _atyp(U.load_df("xl_unlearn", CFG))
if len(xu):
    main = xu[xu["draw"] != CFG["xl_pilot_draw"]]
    print("atypical unlearning - matched rate and accuracies (means over 9 forget sets):")
    display(main.groupby("method")[["matched", "forget_acc", "ref_forget_acc", "test_acc", "ref_test_acc"]].mean().round(4))
xe = _atyp(U.load_df("xl_extinction", CFG))
if len(xe):
    ids = ["kind", "method", "seed", "draw"]
    base = xe[xe["test"] == "base"][ids + ["forget_acc"]].rename(columns={"forget_acc": "before"})
    rf = xe[(xe["test"] == "spontaneous") & (xe["proxy"] == "retain_ft") & np.isclose(pd.to_numeric(xe["level"], errors="coerce"), CFG["sr_epochs"]) & (xe["bn"] == "update")]
    rf = rf.merge(base, on=ids).assign(recovery=lambda d: d["forget_acc"] - d["before"])
    print("retain fine-tuning (5 epochs): forget accuracy after minus before:")
    display(rf.groupby("method")["recovery"].agg(["mean", "count"]).round(4))
hb = U.load_df("headbias", CFG)
if len(hb):
    print("head-bias check v2 (class level), means:")
    display(hb.groupby(["kind", "method"])[["bias_restore_forget_acc", "shift_recall_fpr1", "shift_fpr_fpr1", "shift_recall_fpr5"]].mean().round(3))

## Next step

Save a version, then re-run **notebook 08** (CPU) with `ext-09-atypical` attached as well.

## Name of this run's output

Title this notebook **`ext-09-atypical`** and save its output as a dataset with the same name.

| Output name | Contains | Attached by |
|---|---|---|
| `ext-09-atypical` | `headbias` (v2), atypical `xl_*` results, `xl_calibration_atypical.json`, 10 retrained + 50 unlearned checkpoints (≈ 1.3 GB) | notebook 08 |